# Projet final — pipeline complet

Ce notebook exécute les six phases dans l’ordre : EDA, baseline, E1, E2 (LOADO + sibling), E3 et XAI. Le bloc de helpers du début est partagé par toutes les phases.

# 01 - Phase 1 : Analyse exploratoire des données

## Question de recherche
Est-ce qu’un modèle entraîné sur certaines familles de DDoS peut reconnaître une stratégie de DDoS qu’il n’a jamais vue ?  Le jeu de données est CIC-DDoS2019, publié sous forme de 17 fichiers parquet avec 77 caractéristiques de flux et 17 étiquettes fines.

## Hypothèses
- **H1** - La classification multiclasses est plus difficile que la classification binaire.
- **H2** - La généralisation à une famille non observée dépend fortement de la famille laissée de côté : certains mécanismes sont reconnaissables à partir de la structure du flux partagée, d’autres ne sont que mémorisés.
- **H3** - Une fraction des 77 caractéristiques conserve l’essentiel de la performance, et un ensemble de caractéristiques plus petit réduit le coût d’inférence.


## Pourquoi ce notebook
Deux artefacts du jeu de données peuvent donner l’impression d’un bon résultat : des lignes dupliquées exactes et la séparation en fichiers `-training` / `-testing`.  La séparation est organisée par *mécanisme* d’attaque, pas au hasard, donc l’utiliser mesure la reconnaissance de mécanisme plutôt que la généralisation.  Ce notebook quantifie les deux artefacts avant qu’aucun modèle ne soit entraîné.


In [ ]:
import json
import time
import warnings
from pathlib import Path
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix as sk_confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

ROOT = Path.cwd()
if not (ROOT / "dataset").exists():
    ROOT = ROOT.parent
DATA = ROOT / "dataset"
RES = ROOT / "results"
FIG = ROOT / "figures"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

BENIGN = "Benign"
SEEDS = [42, 43, 44]
CAP = 20_000


def load_dataset(data_dir=None):
    files = sorted(Path(data_dir or DATA).glob("*.parquet"))
    frames = []
    for path in files:
        frame = pd.read_parquet(path)
        frame.columns = [str(column).strip() for column in frame.columns]
        frame["src_file"] = path.name
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    raw["Label"] = raw["Label"].astype(str).str.strip().replace({"UDP-lag": "UDPLag"})
    features = [column for column in raw.columns if column not in ("Label", "src_file")]
    X = raw[features].astype("float64")
    y_family = raw["Label"].astype("string").rename("family")
    y_binary = (y_family != BENIGN).astype("int8").rename("binary")
    meta = pd.DataFrame(
        {"src_file": raw["src_file"].astype("string"), "label": y_family.to_numpy()},
        index=X.index,
    )
    return X, y_family, y_binary, meta


def dedupe(X, y, meta=None):
    key = pd.Series(pd.util.hash_pandas_object(X, index=False).to_numpy(), index=X.index)
    grouped = y.groupby(key, sort=False)
    labels = grouped.first()
    n_labels = grouped.nunique()
    conflicts = n_labels[n_labels > 1].index
    if len(conflicts):
        subset = pd.DataFrame({"hash": key, "y": y}).loc[lambda frame: frame["hash"].isin(conflicts)]
        majority = subset.groupby("hash")["y"].agg(
            lambda values: sorted(values.value_counts().loc[lambda counts: counts == counts.max()].index)[0]
        )
        labels.loc[conflicts] = majority.reindex(conflicts)
    keep = ~key.duplicated(keep="first")
    X_out = X.loc[keep].reset_index(drop=True)
    y_out = pd.Series(labels.reindex(key[keep]).to_numpy(), index=X_out.index, name=y.name)
    X_out.attrs["dedupe"] = {
        "rows_in": int(len(X)),
        "rows_out": int(len(X_out)),
        "rows_removed": int(len(X) - len(X_out)),
        "rows_in_duplicate_groups": int(key.duplicated(keep=False).sum()),
        "conflicting_hashes": int(len(conflicts)),
    }
    if meta is None:
        return X_out, y_out
    meta_out = meta.loc[keep].reset_index(drop=True)
    return X_out, y_out, meta_out


def load_deduped(data_dir=None):
    X, y_family, _, meta = load_dataset(data_dir)
    X, y_family, meta = dedupe(X, y_family, meta)
    y_binary = (y_family != BENIGN).astype("int8").rename("binary")
    return X, y_family, y_binary, meta


def group_attack_families(y):
    return y.astype(str).str.replace(r"^DrDoS_", "", regex=True).astype("string")


def split(X, y, seed=42, test_size=0.2):
    return train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=seed, shuffle=True
    )


def subsample(X, y, cap=CAP, seed=42):
    rng = np.random.default_rng(seed)
    picks = []
    for _, index in y.groupby(y, sort=False).groups.items():
        index = np.asarray(index)
        if len(index) > cap:
            index = rng.choice(index, size=cap, replace=False)
        picks.append(index)
    chosen = np.concatenate(picks)
    rng.shuffle(chosen)
    return X.loc[chosen].reset_index(drop=True), y.loc[chosen].reset_index(drop=True)


def purge_family(X, y, family):
    held = y == family
    keep = ~held
    return (
        X.loc[keep].reset_index(drop=True),
        y.loc[keep].reset_index(drop=True),
        X.loc[held].reset_index(drop=True),
        y.loc[held].reset_index(drop=True),
    )


def drop_constants(X_train, X_test):
    keep = X_train.columns[X_train.nunique() > 1]
    return X_train[keep].copy(), X_test[keep].copy(), list(keep)


def build_pipeline(clf):
    return Pipeline([("scale", StandardScaler()), ("clf", clf)])


def model(name, seed=42, n_estimators=300):
    if name == "dummy":
        return DummyClassifier(strategy="stratified", random_state=seed)
    if name == "logreg":
        return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed)
    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=n_estimators,
            class_weight="balanced",
            n_jobs=-1,
            random_state=seed,
        )
    raise ValueError(name)


def make_models(seed=42, names=None, n_estimators=300):
    names = names or ["dummy", "logreg", "random_forest"]
    return {name: build_pipeline(model(name, seed, n_estimators)) for name in names}


def fpr_at_tpr95(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return float(np.interp(0.95, tpr, fpr))


def evaluate(y_true, y_pred, y_proba, classes):
    classes = list(classes)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    proba = np.asarray(y_proba)
    if proba.ndim == 1:
        proba = np.column_stack([1.0 - proba, proba])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, average=None, zero_division=0
    )
    rows = []
    for index, label in enumerate(classes):
        binary_true = (y_true == label).astype(int)
        binary_pred = (y_pred == label).astype(int)
        tp = int(((binary_true == 1) & (binary_pred == 1)).sum())
        fp = int(((binary_true == 0) & (binary_pred == 1)).sum())
        tn = int(((binary_true == 0) & (binary_pred == 0)).sum())
        fn = int(((binary_true == 1) & (binary_pred == 0)).sum())
        row = {
            "class": label,
            "precision": float(precision[index]),
            "recall": float(recall[index]),
            "f1": float(f1[index]),
            "support": int(support[index]),
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
            "fpr": float(fp / (fp + tn)) if (fp + tn) else np.nan,
            "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "fpr_at_tpr95": np.nan,
        }
        if 0 < binary_true.sum() < len(binary_true):
            row["pr_auc"] = float(average_precision_score(binary_true, proba[:, index]))
            row["roc_auc"] = float(roc_auc_score(binary_true, proba[:, index]))
            row["fpr_at_tpr95"] = fpr_at_tpr95(binary_true, proba[:, index])
        rows.append(row)
    per_class = pd.DataFrame(rows).set_index("class", drop=False)
    per_class.index.name = None
    if len(classes) == 2:
        positive = per_class.iloc[1]
        pr_auc = float(positive["pr_auc"])
        roc_auc = float(positive["roc_auc"])
        fpr95 = float(positive["fpr_at_tpr95"])
        fpr = float(positive["fpr"])
    else:
        pr_auc = float(np.nanmean(per_class["pr_auc"]))
        roc_auc = float(np.nanmean(per_class["roc_auc"]))
        fpr95 = float(np.nanmean(per_class["fpr_at_tpr95"]))
        fpr = float(np.nanmean(per_class["fpr"]))
    return {
        "n_test": int(len(y_true)),
        "n_classes": len(classes),
        "macro_f1": float(f1_score(y_true, y_pred, labels=classes, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=classes, average="weighted", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "fpr": fpr,
        "fpr_at_tpr95": fpr95,
        "per_class": per_class,
    }


def confusion_matrix(y_true, y_pred, labels):
    matrix = sk_confusion_matrix(y_true, y_pred, labels=list(labels))
    return pd.DataFrame(
        matrix,
        index=pd.Index(list(labels), name="true"),
        columns=pd.Index(list(labels), name="pred"),
    )


def fit_and_score(pipeline, X_train, y_train, X_test, y_test):
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    y_pred = pipeline.predict(X_test)
    predict_seconds = time.perf_counter() - start
    y_proba = pipeline.predict_proba(X_test)
    classes = list(pipeline.named_steps["clf"].classes_)
    metrics = evaluate(y_test, y_pred, y_proba, classes)
    return {
        "metrics": metrics,
        "y_pred": np.asarray(y_pred),
        "classes": classes,
        "fit_seconds": float(fit_seconds),
        "predict_seconds": float(predict_seconds),
        "n_train": int(len(y_train)),
        "n_features_in": int(X_train.shape[1]),
        "n_features_effective": int(X_train.shape[1]),
    }


def metrics_row(metrics, **extra):
    row = {
        "n_test": metrics["n_test"],
        "n_classes": metrics["n_classes"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "accuracy": metrics["accuracy"],
        "pr_auc": metrics["pr_auc"],
        "roc_auc": metrics["roc_auc"],
        "fpr": metrics["fpr"],
        "fpr_at_tpr95": metrics["fpr_at_tpr95"],
    }
    row.update(extra)
    return row


def median_latency_ms(pipeline, X_batch, repeats=5):
    pipeline.predict(X_batch)
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        pipeline.predict(X_batch)
        timings.append((time.perf_counter() - start) * 1000.0)
    return float(np.median(timings))


def run_baseline(seed=42):
    X, y_family, _, _ = load_deduped()
    X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=seed)
    X_train, X_test, features = drop_constants(X_train, X_test)
    y_train = (y_family_train != BENIGN).astype("int8")
    y_test = (y_family_test != BENIGN).astype("int8")
    rows, per_class_rows = [], []
    for name, pipeline in make_models(seed=seed).items():
        result = fit_and_score(pipeline, X_train, y_train, X_test, y_test)
        metrics = result["metrics"]
        rows.append(
            metrics_row(
                metrics,
                model=name,
                model_display=name,
                seed=seed,
                fit_seconds=result["fit_seconds"],
                predict_seconds=result["predict_seconds"],
                n_train=result["n_train"],
                n_features_effective=len(features),
            )
        )
        per_class = metrics["per_class"].copy()
        per_class.insert(0, "model", name)
        per_class.insert(1, "seed", seed)
        per_class_rows.append(per_class)
        confusion_matrix(y_test, result["y_pred"], [0, 1]).to_csv(
            RES / f"baseline_confusion_{name}.csv"
        )
        print(f"{name:14s} macro-F1={metrics['macro_f1']:.4f} FPR={metrics['fpr']:.4f}")
    pd.DataFrame(rows).to_csv(RES / "baseline_binary.csv", index=False)
    pd.concat(per_class_rows, ignore_index=True).to_csv(
        RES / "baseline_binary_per_class.csv", index=False
    )
    return pd.DataFrame(rows)


def run_exp1(seed=42):
    X, y_family, _, _ = load_deduped()
    X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=seed)
    X_train, X_test, _ = drop_constants(X_train, X_test)
    X_train_np, X_test_np, _ = drop_constants(
        X_train.drop(columns=["Protocol"]), X_test.drop(columns=["Protocol"])
    )
    y_binary_train = (y_family_train != BENIGN).astype("int8")
    y_binary_test = (y_family_test != BENIGN).astype("int8")
    y_group_train = group_attack_families(y_family_train)
    y_group_test = group_attack_families(y_family_test)
    classes = sorted(y_group_train.unique())
    tasks = [
        ("binary", X_train, X_test, y_binary_train, y_binary_test, [0, 1]),
        ("multiclass", X_train, X_test, y_group_train, y_group_test, classes),
        ("multiclass_no_protocol", X_train_np, X_test_np, y_group_train, y_group_test, classes),
    ]
    rows, per_class_rows = [], []
    for task, Xtr, Xte, ytr, yte, task_classes in tasks:
        for name, pipeline in make_models(seed=seed).items():
            result = fit_and_score(pipeline, Xtr, ytr, Xte, yte)
            metrics = result["metrics"]
            rows.append(
                metrics_row(
                    metrics,
                    task=task,
                    model=name,
                    model_display=name,
                    seed=seed,
                    fit_seconds=result["fit_seconds"],
                    predict_seconds=result["predict_seconds"],
                    n_train=result["n_train"],
                    n_features_in=result["n_features_in"],
                    n_features_effective=result["n_features_effective"],
                )
            )
            per_class = metrics["per_class"].copy()
            per_class.insert(0, "task", task)
            per_class.insert(1, "model", name)
            per_class.insert(2, "seed", seed)
            per_class_rows.append(per_class)
            confusion_matrix(yte, result["y_pred"], task_classes).to_csv(
                RES / f"exp1_confusion_{task}_{name}.csv"
            )
            print(f"{task:24s} {name:14s} macro-F1={metrics['macro_f1']:.4f}")
    pd.DataFrame(rows).to_csv(RES / "exp1.csv", index=False)
    pd.concat(per_class_rows, ignore_index=True).to_csv(RES / "exp1_per_class.csv", index=False)
    return pd.DataFrame(rows)


def balanced_test(held_X, held_y, X_known_test, y_known_test, seed):
    benign_index = y_known_test.index[y_known_test == BENIGN].to_numpy()
    n = min(len(held_X), len(benign_index))
    rng = np.random.default_rng(seed)
    held_positions = rng.choice(len(held_X), size=n, replace=False) if len(held_X) > n else np.arange(len(held_X))
    benign_positions = rng.choice(benign_index, size=n, replace=False) if len(benign_index) > n else benign_index
    X_attack = held_X.iloc[held_positions].reset_index(drop=True)
    X_benign = X_known_test.loc[benign_positions].reset_index(drop=True)
    X_eval = pd.concat([X_attack, X_benign], ignore_index=True)
    y_eval = np.r_[np.ones(len(X_attack), dtype=np.int8), np.zeros(len(X_benign), dtype=np.int8)]
    return X_eval, y_eval, X_attack


def run_exp2():
    X, y_family, _, meta = load_deduped()
    counts = y_family.value_counts()
    families = sorted(label for label in counts.index if label != BENIGN and counts[label] >= 1_000)
    audit = (
        meta.loc[meta["label"].isin(families)]
        .groupby(["label", "src_file"], observed=True)
        .size()
        .reset_index(name="rows")
        .rename(columns={"label": "family"})
    )
    audit.to_csv(RES / "exp2_purge_audit.csv", index=False)
    rows, per_class_rows, confusion_rows = [], [], []
    diagnostic_seed = SEEDS[0]
    for seed in SEEDS:
        for family in families:
            X_pool, y_pool, held_X, held_y = purge_family(X, y_family, family)
            X_train, X_test, y_train, y_test = split(X_pool, y_pool, seed=seed)
            X_train, X_test, features = drop_constants(X_train, X_test)
            X_train, y_train = subsample(X_train, y_train, cap=CAP, seed=seed)
            y_binary_train = (y_train != BENIGN).astype("int8")
            held_X = held_X[features].copy()
            X_eval, y_eval, X_attack = balanced_test(held_X, held_y, X_test, y_test, seed)
            for name, pipeline in make_models(seed=seed).items():
                result = fit_and_score(pipeline, X_train, y_binary_train, X_eval, y_eval)
                metrics = result["metrics"]
                rows.append(
                    metrics_row(
                        metrics,
                        family=family,
                        seed=seed,
                        model=name,
                        model_display=name,
                        n_family_available=int(len(held_X)),
                        n_test_heldout=int(y_eval.sum()),
                        n_test_benign=int((y_eval == 0).sum()),
                        n_train=result["n_train"],
                        n_features_effective=len(features),
                        fit_seconds=result["fit_seconds"],
                        predict_seconds=result["predict_seconds"],
                        recall_family=float(metrics["per_class"].loc[1, "recall"]),
                    )
                )
                per_class = metrics["per_class"].copy()
                per_class.insert(0, "family", family)
                per_class.insert(1, "model", name)
                per_class.insert(2, "seed", seed)
                per_class_rows.append(per_class)
                confusion_matrix(y_eval, result["y_pred"], [0, 1]).to_csv(
                    RES / f"exp2_confusion_{family}_{name}_s{seed}.csv"
                )
                print(f"{family:16s} {name:14s} seed={seed} macro-F1={metrics['macro_f1']:.3f}")
            if seed == diagnostic_seed:
                known_classes = sorted(y_train.unique().tolist())
                X_diag, y_diag = subsample(X_train, y_train, cap=5_000, seed=seed)
                diag_pipeline = build_pipeline(
                    RandomForestClassifier(
                        n_estimators=100,
                        class_weight="balanced",
                        n_jobs=-1,
                        random_state=seed,
                    )
                )
                diag_pipeline.fit(X_diag, y_diag)
                predicted = diag_pipeline.predict(X_attack)
                table = pd.crosstab(
                    pd.Series([family] * len(predicted)),
                    pd.Series(predicted),
                ).reindex(columns=known_classes, fill_value=0)
                table.index.name = "held_out_family"
                table.columns.name = "predicted_known_family"
                part = table.stack().rename("count").reset_index()
                part.insert(0, "seed", seed)
                confusion_rows.append(part)
    results = pd.DataFrame(rows)
    results.to_csv(RES / "exp2.csv", index=False)
    pd.concat(per_class_rows, ignore_index=True).to_csv(RES / "exp2_per_class.csv", index=False)
    if confusion_rows:
        pd.concat(confusion_rows, ignore_index=True).to_csv(
            RES / "exp2_confusion_to.csv", index=False
        )
    return results


def run_sibling(seed=42):
    X, y_family, _, _ = load_deduped()
    pairs = [
        ("LDAP", ["LDAP", "DrDoS_LDAP"]),
        ("DrDoS_LDAP", ["DrDoS_LDAP", "LDAP"]),
        ("MSSQL", ["MSSQL", "DrDoS_MSSQL"]),
        ("DrDoS_MSSQL", ["DrDoS_MSSQL", "MSSQL"]),
        ("UDP", ["UDP", "DrDoS_UDP"]),
        ("DrDoS_UDP", ["DrDoS_UDP", "UDP"]),
    ]
    rows = []
    for target, purge_labels in pairs:
        held = y_family == target
        keep = ~y_family.isin(purge_labels)
        X_pool = X.loc[keep].reset_index(drop=True)
        y_pool = y_family.loc[keep].reset_index(drop=True)
        held_X = X.loc[held].reset_index(drop=True)
        held_y = y_family.loc[held].reset_index(drop=True)
        X_train, X_test, y_train, y_test = split(X_pool, y_pool, seed=seed)
        X_train, X_test, features = drop_constants(X_train, X_test)
        held_X = held_X[features].copy()
        X_train, y_train = subsample(X_train, y_train, cap=CAP, seed=seed)
        y_binary_train = (y_train != BENIGN).astype("int8")
        X_eval, y_eval, _ = balanced_test(held_X, held_y, X_test, y_test, seed)
        pipeline = make_models(seed=seed, names=["random_forest"])["random_forest"]
        result = fit_and_score(pipeline, X_train, y_binary_train, X_eval, y_eval)
        metrics = result["metrics"]
        rows.append(
            metrics_row(
                metrics,
                target_family=target,
                purged_labels=", ".join(purge_labels),
                seed=seed,
                n_test_heldout=int(y_eval.sum()),
                n_test_benign=int((y_eval == 0).sum()),
                recall_target=float(metrics["per_class"].loc[1, "recall"]),
            )
        )
        print(f"{target:14s} purge={purge_labels!s:35s} macro-F1={metrics['macro_f1']:.4f}")
    out = pd.DataFrame(rows)
    out.to_csv(RES / "exp2_sibling_check.csv", index=False)
    return out


def run_exp3():
    X, y_family, _, _ = load_deduped()
    variants = {"all": None, "75pct": 58, "50pct": 39, "25pct": 20, "top10": 10}
    rows, ranking_tables, selected_rows = [], [], []
    for seed in SEEDS:
        X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=seed)
        X_train, X_test, features = drop_constants(X_train, X_test)
        X_train, y_family_train = subsample(X_train, y_family_train, cap=CAP, seed=seed)
        tasks = {
            "binary": (
                (y_family_train != BENIGN).astype("int8"),
                (y_family_test != BENIGN).astype("int8"),
                [0, 1],
            ),
            "multiclass": (
                group_attack_families(y_family_train),
                group_attack_families(y_family_test),
                sorted(group_attack_families(y_family_train).unique()),
            ),
        }
        for task, (y_train, y_test, classes) in tasks.items():
            mi = pd.Series(
                mutual_info_classif(X_train, y_train, discrete_features=False, random_state=seed),
                index=X_train.columns,
                name="mutual_info",
            )
            forest = RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                n_jobs=-1,
                random_state=seed,
            )
            forest.fit(X_train, y_train)
            importance = pd.Series(forest.feature_importances_, index=X_train.columns, name="rf_importance")
            ranking = pd.concat([mi, importance], axis=1)
            ranking["mi_rank"] = ranking["mutual_info"].rank(ascending=False, method="min").astype(int)
            ranking["rf_rank"] = ranking["rf_importance"].rank(ascending=False, method="min").astype(int)
            ranking.insert(0, "task", task)
            ranking.insert(0, "seed", seed)
            ranking_tables.append(ranking.reset_index().rename(columns={"index": "feature"}))
            for variant, k in variants.items():
                if k is None:
                    pipeline = build_pipeline(
                        RandomForestClassifier(
                            n_estimators=300,
                            class_weight="balanced",
                            n_jobs=-1,
                            random_state=seed,
                        )
                    )
                else:
                    pipeline = Pipeline(
                        [
                            (
                                "select",
                                SelectKBest(
                                    score_func=partial(
                                        mutual_info_classif,
                                        discrete_features=False,
                                        random_state=seed,
                                    ),
                                    k=k,
                                ),
                            ),
                            ("scale", StandardScaler()),
                            (
                                "clf",
                                RandomForestClassifier(
                                    n_estimators=300,
                                    class_weight="balanced",
                                    n_jobs=-1,
                                    random_state=seed,
                                ),
                            ),
                        ]
                    )
                result = fit_and_score(pipeline, X_train, y_train, X_test, y_test)
                latency = median_latency_ms(pipeline, X_test.iloc[:1_000])
                n_effective = k if k is not None else len(features)
                rows.append(
                    metrics_row(
                        result["metrics"],
                        task=task,
                        variant=variant,
                        k_requested=k,
                        model="random_forest",
                        seed=seed,
                        n_train=result["n_train"],
                        n_features_in=result["n_features_in"],
                        n_features_effective=n_effective,
                        fit_seconds=result["fit_seconds"],
                        predict_seconds=result["predict_seconds"],
                        latency_ms_1000=latency,
                        throughput_flows_s=1_000.0 / (latency / 1000.0),
                    )
                )
                confusion_matrix(y_test, result["y_pred"], classes).to_csv(
                    RES / f"exp3_confusion_{task}_{variant}_s{seed}.csv"
                )
                if k is not None:
                    chosen = X_train.columns[pipeline.named_steps["select"].get_support()]
                    selected_rows.extend(
                        {
                            "task": task,
                            "variant": variant,
                            "seed": seed,
                            "feature": feature,
                            "rank": rank,
                        }
                        for rank, feature in enumerate(chosen, start=1)
                    )
                print(f"{task:10s} {variant:6s} k={n_effective:3d} macro-F1={result['metrics']['macro_f1']:.4f}")
    pd.DataFrame(rows).to_csv(RES / "exp3.csv", index=False)
    pd.concat(ranking_tables, ignore_index=True).to_csv(RES / "exp3_feature_ranking.csv", index=False)
    pd.DataFrame(selected_rows).to_csv(RES / "exp3_selected_features.csv", index=False)
    return pd.DataFrame(rows)


In [ ]:
X, y_family, y_binary, meta = load_dataset()
print(f"lignes      : {len(X):,}")
print(f"caractéristiques : {X.shape[1]}")
print(f"familles    : {y_family.nunique()}  (Benign + {y_family.nunique() - 1} familles d’attaque)")
print(f"fichiers    : {meta['src_file'].nunique()}")
display(meta.head())
display(X.head(3))

In [ ]:
counts = y_family.value_counts().rename("rows").to_frame()
counts["share_pct"] = (100 * counts["rows"] / counts["rows"].sum()).round(3)
counts["kind"] = np.where(counts.index == BENIGN, "Benign", "Attack")
display(counts)
counts.to_csv(RES / "eda_family_counts.csv")

fig, ax = plt.subplots(figsize=(10, 6))
order = counts.index[::-1]
colors = ["#4c72b0" if label == BENIGN else "#c44e52" for label in order]
ax.barh(order, counts.loc[order, "rows"], color=colors)
ax.set_xscale("log")
ax.set_xlabel("lignes (base 10 logarithmique)")
ax.set_title("Déséquilibre de classes: 17 familles")
for i, label in enumerate(order):
    ax.text(counts.loc[label, "rows"] * 1.08, i, f"{counts.loc[label, 'rows']:,}", va="center", fontsize=8)
ax.margins(x=0.18)
fig.tight_layout()
fig.savefig(FIG / "01_eda_family_imbalance.png", bbox_inches="tight")
plt.show()

In [ ]:
protocol = pd.crosstab(y_family, X["Protocol"])
protocol_share = protocol.div(protocol.sum(axis=1), axis=0)
display(protocol)
display((100 * protocol_share).round(2))

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(
    protocol_share,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={"label": "Part de de la famille"},
    ax=ax,
)
ax.set_title("Famille × Protocole: le protocole est presque déterministe par famille")
ax.set_xlabel("Code du protocole (0, 6 = TCP, 17 = UDP)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIG / "01_eda_family_protocol.png", bbox_inches="tight")
plt.show()

print("Syn TCP :", f"{protocol_share.loc['Syn', 6]:.4f}")
print("MSSQL UDP :", f"{protocol_share.loc['MSSQL', 17]:.4f}")
print("UDP UDP :", f"{protocol_share.loc['UDP', 17]:.4f}")

In [ ]:
meta_audit = meta.copy()
meta_audit["partition"] = meta_audit["src_file"].str.extract(r"-(training|testing)\.parquet")[0]
partition = pd.crosstab(y_family, meta_audit["partition"])
display(partition)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    partition,
    annot=True,
    fmt="d",
    cmap="OrRd",
    cbar_kws={"label": "lignes"},
    ax=ax,
)
ax.set_title("Découpage des fichiers publiés par mécanisme d'attaque")
ax.set_xlabel("partition publiée")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(FIG / "01_eda_file_partition.png", bbox_inches="tight")
plt.show()

file_family = pd.crosstab(meta_audit["src_file"], y_family)
fig, ax = plt.subplots(figsize=(15, 8))
annot = file_family.map(lambda value: f"{value:,}" if value else "")
sns.heatmap(
    np.log1p(file_family),
    annot=annot,
    fmt="",
    cmap="viridis",
    cbar_kws={"label": "log1p(lignes)"},
    ax=ax,
)
ax.set_title("Quel fichier contient quelle famille (comptages annotés)")
ax.set_xlabel("Fichiers")
ax.set_ylabel("Familles")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG / "01_eda_file_family_matrix.png", bbox_inches="tight")
plt.show()

only_testing = sorted(
    label for label in partition.index if partition.loc[label].get("training", 0) == 0
)
only_training = sorted(
    label for label in partition.index if partition.loc[label].get("testing", 0) == 0
)
both = sorted(
    label
    for label in partition.index
    if partition.loc[label].get("training", 0) > 0
    and partition.loc[label].get("testing", 0) > 0
)
print("seulement dans les fichiers -testing :", only_testing)
print("seulement dans les fichiers -training :", only_training)
print("présent dans les 2:", both)

In [ ]:
# Audit de fuite sur le tableau brut concaténé, avant tout split.
feature_names = list(X.columns)
raw_hash = pd.util.hash_pandas_object(X, index=False)
hash_meta = pd.DataFrame({"hash": raw_hash, "src_file": meta["src_file"].to_numpy(), "family": y_family.to_numpy()})

rows_in_dup_groups = int(raw_hash.duplicated(keep=False).sum())
hashes_in_multiple_files = int(
    hash_meta.groupby("hash")["src_file"].nunique().gt(1).sum()
)
labels_per_hash = hash_meta.groupby("hash")["family"].nunique()
conflicting_hashes = int(labels_per_hash.gt(1).sum())
rows_in_conflicts = int(hash_meta["hash"].isin(labels_per_hash[labels_per_hash > 1].index).sum())

X_deduped, y_deduped = dedupe(X, y_family)
dedupe_stats = X_deduped.attrs["dedupe"]

audit = pd.DataFrame(
    {
        "check": [
            "lignes dans des groupes de doublons exacts",
            "empreintes de caractéristiques présentes dans >1 fichier source",
            "empreintes de caractéristiques avec étiquettes conflictuelles",
            "lignes impliquées dans des empreintes de caractéristiques avec étiquettes conflictuelles",
            "lignes supprimées par le dédoublonnage",
            "lignes après le dédoublonnage",
        ],
        "value": [
            rows_in_dup_groups,
            hashes_in_multiple_files,
            conflicting_hashes,
            rows_in_conflicts,
            dedupe_stats["rows_removed"],
            dedupe_stats["rows_out"],
        ],
    }
)
display(audit)
audit.to_csv(RES / "eda_leakage_audit.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(audit["check"][::-1], audit["value"][::-1], color="#55a868")
ax.set_xscale("log")
ax.set_xlabel("nombre (échelle logarithmique)")
ax.set_title("Audit de fuite avant le split")
for i, value in enumerate(audit["value"][::-1]):
    ax.text(value * 1.05, i, f"{value:,}", va="center", fontsize=9)
ax.margins(x=0.2)
fig.tight_layout()
fig.savefig(FIG / "01_eda_leakage_audit.png", bbox_inches="tight")
plt.show()

In [ ]:
# Choix des six caractéristiques au signal univarié le plus fort, sur un
# échantillon stratifié (EDA uniquement ; aucune décision de sélection de modèle ici).
sample_parts = []
for _, group in X.assign(_family=y_family.to_numpy()).groupby("_family", sort=False):
    sample_parts.append(group.sample(n=min(1500, len(group)), random_state=42))
sample = pd.concat(sample_parts, ignore_index=True)
mi = mutual_info_classif(
    sample[feature_names],
    sample["_family"],
    discrete_features=False,
    random_state=42,
)
mi = pd.Series(mi, index=feature_names).sort_values(ascending=False)
display(mi.head(20).rename("mutual_info").to_frame())
mi.to_csv(RES / "eda_mutual_info.csv", header=["mutual_info"])
top6 = mi.head(6).index.tolist()
print("top 6 :", top6)

In [ ]:
plot_sample = X[top6].sample(n=min(20_000, len(X)), random_state=42).copy()
plot_sample["family"] = y_family.loc[plot_sample.index].to_numpy()
order = y_family.value_counts().index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
for ax, feature in zip(axes.ravel(), top6):
    sns.boxplot(
        data=plot_sample,
        x="family",
        y=feature,
        order=order,
        showfliers=False,
        color="#4c72b0",
        ax=ax,
    )
    ax.set_yscale("symlog")
    ax.set_title(feature)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=90, labelsize=8)
fig.suptitle("Distributions par famille des 6 meilleures caractéristiques EDA", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "01_eda_top_feature_distributions.png", bbox_inches="tight")
plt.show()

## Ce que ce notebook établit

**Observation.** Les 17 étiquettes sont extrêmement déséquilibrées (NTP 121k vs WebDDoS 49).  Le protocole détermine presque la famille pour Syn, MSSQL et UDP.  Les sept familles `DrDoS_*` n’apparaissent que dans les fichiers `-testing`; LDAP, MSSQL, NetBIOS, Portmap et UDP n’apparaissent que dans les fichiers `-training`.  Il y a 21 347 lignes dans des groupes de doublons exacts, 8 858 empreintes de caractéristiques qui apparaissent dans plus d’un fichier source, et 5 289 empreintes dont les doublons ne sont pas d’accord sur l’étiquette.


**Interprétation ML.** Une séparation aléatoire après déduplication est nécessaire.  Le découpage publié mettrait des mécanismes entiers d’un côté, donc un modèle pourrait bien scorer en reconnaissant le fichier/mécanisme plutôt que l’attaque.  Les doublons exacts mettraient également la même ligne dans train et test.  La déduplication par hash de caractéristiques supprime les deux raccourcis.

**Interprétation cyber.** `Protocol` est proche d’une table de correspondance pour plusieurs familles.  Un score élevé en multiclasses peut donc mesurer principalement la reconnaissance du protocole.  L’expérience 1 (phase 3) répète la tâche multiclasses sans `Protocol` pour quantifier ce raccourci.  Des vecteurs de caractéristiques identiques avec des étiquettes différentes sont une vraie limitation : certains flux de réflexion et d’exploitation sont indistinguables au niveau du flux, donc le bruit d’étiquette est irréductible pour ces paires.

**Limite.** L’analyse EDA est basée sur un échantillon et utilise l’information mutuelle univariée. Elle ne représente pas la sélection de l’expérience 3 (phase 5), qui est ajustée uniquement sur le pli d’entraînement. Le jeu de données est une capture de laboratoire de 2019 et ne représente pas le trafic de production actuel, les charges utiles chiffrées ou la dérive conceptuelle.


# 02 - Phase 2 : Références (Benign / DDoS)

## Question
Avant de comparer quoi que ce soit de complexe : quelle est la performance d’un dummy, d’un modèle linéaire et d’un ensemble d’arbres sur la tâche binaire, avec la séparation sans fuite ?

## Protocole
- CIC-DDoS2019 après déduplication : 77 caractéristiques, répartition en 80/20
- `DummyClassifier(strategy="stratified")` est le palier minimal.
- `LogisticRegression(max_iter=2000, class_weight="balanced")` est la référence linéaire.
- `RandomForestClassifier(n_estimators=300, class_weight="balanced")` est la référence non linéaire.
- Les colonnes constantes sont retirées sur le pli d’entraînement ; la mise à l’échelle est dans la pipeline et ne s’applique qu’au train.
- Métriques : Macro-F1, Weighted-F1, précision/rappel/F1 par classe, PR-AUC, ROC-AUC, FPR à 95 % de rappel. L’accuracy est affichée mais n’est pas utilisée pour conclure.

Ce notebook entraîne lui-même le dummy, la régression logistique et la forêt aléatoire, puis écrit `results/baseline_binary.csv`. Aucun module externe ni CLI.


In [ ]:
baseline = run_baseline(seed=42)
results = pd.read_csv(RES / "baseline_binary.csv")
per_class = pd.read_csv(RES / "baseline_binary_per_class.csv")
display(results[["model", "macro_f1", "weighted_f1", "accuracy", "pr_auc", "roc_auc", "fpr", "fpr_at_tpr95", "fit_seconds", "n_features_effective"]].round(4))
print("sklearn:", results["sklearn_version"].unique().tolist())

In [ ]:
metrics_long = results.melt(
    id_vars=["model"],
    value_vars=["macro_f1", "weighted_f1", "accuracy", "pr_auc", "roc_auc", "fpr"],
    var_name="metric",
    value_name="value",
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=metrics_long, x="metric", y="value", hue="model", ax=ax)
ax.set_ylim(0, 1.02)
ax.set_title("Baseline binaire : le palier se sépare immédiatement des modèles")
ax.set_xlabel("")
ax.legend(title="", ncol=3, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "02_baseline_metrics.png", bbox_inches="tight")
plt.show()

display(
    results[["model", "macro_f1", "pr_auc", "roc_auc", "fpr", "fpr_at_tpr95"]]
    .set_index("model")
    .round(4)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, model in zip(axes, ["dummy", "logreg", "random_forest"]):
    matrix = pd.read_csv(RES / f"baseline_confusion_{model}.csv", index_col=0)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(model)
    ax.set_xlabel("prédit")
    ax.set_ylabel("réel")
fig.suptitle("Matrice de confusion - binaire Benign / DDoS", y=1.03)
fig.tight_layout()
fig.savefig(FIG / "02_baseline_confusion.png", bbox_inches="tight")
plt.show()

In [ ]:
# Vérification du dummy binaire : un tirage stratifié doit rester proche du hasard.
X, y_family, _, _ = load_deduped()
X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=42)
y_train = (y_family_train != BENIGN).astype("int8")
y_test = (y_family_test != BENIGN).astype("int8")

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)
pred = dummy.predict(X_test)
prior = y_train.value_counts(normalize=True)
print(f"classes : {y_train.nunique()}")
print(f"1 / n_classes : {1 / y_train.nunique():.4f}")
print(f"sum(prior^2) (accuracy) : {(prior**2).sum():.4f}")
print(f"accuracy du dummy : {accuracy_score(y_test, pred):.4f}")
print(f"Macro-F1 du dummy : {f1_score(y_test, pred, average='macro', zero_division=0):.4f}")
print("Les étiquettes sont alignées : le dummy reste au niveau du hasard binaire.")


## Interprétation des résultats

**Vérification.** Le dummy binaire atteint un Macro-F1 ≈ 0,50 et une accuracy ≈ 0,65.  Le Macro-F1 est cohérent avec un tirage aléatoire à deux classes ; l’accuracy plus élevée vient du déséquilibre Benign/DDoS et ne doit pas être lue comme une performance.  Les étiquettes sont donc alignées.

**Observation.** La régression logistique et la forêt aléatoire dépassent toutes deux 0,99 de Macro-F1.  La forêt aléatoire est légèrement devant (≈ 0,999) et surtout plus prudente : son FPR tombe à ≈ 0,03 %, contre ≈ 0,26 % pour la régression logistique.  L’écart entre les deux modèles est faible comparé à l’écart qui les sépare du dummy.

**Interprétation ML.** La tâche Benign/DDoS est facile sur ce jeu de données : une frontière linéaire suffit déjà à séparer l’essentiel du trafic.  Un score binaire quasi parfait ne prouve donc pas que le modèle distingue les *types* d’attaques ; il prouve seulement qu’il sépare le trafic bénin du trafic offensif.

**Interprétation cyber.** Un détecteur binaire avec un FPR inférieur à 1 % est utile comme premier étage de triage : il réduit le volume à examiner sans noyer l’analyste.  En revanche, il ne dit pas quelle famille d’attaque est en cours et ne permet donc pas de choisir directement une mesure d’atténuation.  Cette phase ne teste que la détection Benign/DDoS ; elle ne mesure ni l’attribution de famille ni la généralisation à une famille inconnue.

**Limites.** L’ensemble de test est un échantillon aléatoire issu d’une capture de laboratoire de 2019.  Il ne mesure ni la dérive, ni l’évasion adversaire, ni le trafic chiffré.  L’accuracy est affichée, mais les conclusions reposent sur le Macro-F1, le rappel par classe et le FPR.


# 03 — Phase 3 · Expérience 1 — Binaire vs multiclasses (H1, Q2)

## Question
Un détecteur binaire qui semble excellent reste-t-il utile lorsque la tâche consiste à identifier la famille d’attaque ?  C’est exactement l’objet de Q2.

## Hypothèse H1
La classification multiclasses est plus difficile que la classification binaire.

## Protocole
- Mêmes lignes dédupliquées, même split stratifié 80/20 par famille, même graine (42), mêmes trois pipelines que la baseline.  Seule la cible change.
- `binary` : Benign vs DDoS.
- `multiclass` : Benign + 12 types d’attaque CIC-DDoS2019, obtenus en fusionnant `DrDoS_X` avec `X` (13 classes).
- `multiclass_no_protocol` : même cible, `Protocol` retiré.  Protocol est presque déterminant pour la famille ; cette ablation mesure donc la part du score multiclasses qui relève d’une lecture du protocole.
- Ce notebook exécute lui-même les 9 fits et écrit `results/exp1.csv` ; il n’y a ni module ni CLI.


In [ ]:
exp1 = run_exp1(seed=42)
results = pd.read_csv(RES / "exp1.csv")
per_class = pd.read_csv(RES / "exp1_per_class.csv")
display(results[["task", "model", "n_classes", "macro_f1", "weighted_f1", "accuracy", "pr_auc", "roc_auc", "fpr", "fpr_at_tpr95", "fit_seconds", "n_features_effective"]].round(4))

In [ ]:
order = ["binary", "multiclass", "multiclass_no_protocol"]
labels = {
    "binary": "Benign / DDoS",
    "multiclass": "13 classes",
    "multiclass_no_protocol": "13 classes, no Protocol",
}
plot = results.copy()
plot["task_label"] = plot["task"].map(labels)
plot["task_label"] = pd.Categorical(plot["task_label"], categories=[labels[t] for t in order], ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, metric in zip(axes, ["macro_f1", "weighted_f1"]):
    sns.barplot(data=plot, x="task_label", y=metric, hue="model", ax=ax)
    ax.set_ylim(0, 1.02)
    ax.set_title(metric.replace("_", "-").title())
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
    ax.legend(title="", fontsize=8)
fig.suptitle("H1 : mêmes lignes, mêmes modèles, seule la cible change", y=1.03)
fig.tight_layout()
fig.savefig(FIG / "03_exp1_macro_weighted_f1.png", bbox_inches="tight")
plt.show()

display(
    results.pivot(index="task", columns="model", values="macro_f1")
    .reindex(order)
    .round(4)
)

In [ ]:
# Rappel par classe pour le modèle multiclasses le plus fort.
focus = per_class[
    (per_class["task"] == "multiclass") & (per_class["model"] == "random_forest")
].copy()
focus["class"] = focus["class"].astype(str)
focus = focus.sort_values("recall")

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=focus, x="recall", y="class", color="#4c72b0", ax=ax)
ax.set_xlim(0, 1.02)
ax.set_title("Rappel par classe — forêt aléatoire multiclasses")
ax.set_xlabel("rappel")
ax.set_ylabel("")
for i, row in enumerate(focus.itertuples()):
    ax.text(row.recall + 0.01, i, f"{row.recall:.3f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "03_exp1_per_class_recall.png", bbox_inches="tight")
plt.show()

display(focus[["class", "precision", "recall", "f1", "support", "fpr"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = ["binary", "multiclass", "multiclass_no_protocol"]
for ax, task in zip(axes, titles):
    matrix = pd.read_csv(
        RES / f"exp1_confusion_{task}_random_forest.csv", index_col=0
    )
    sns.heatmap(matrix, cmap="Blues", cbar=False, ax=ax)
    ax.set_title(task)
    ax.set_xlabel("prédit")
    ax.set_ylabel("réel")
fig.suptitle("Matrices de confusion — binaire (2×2) et multiclasses (13×13), forêt aléatoire", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "03_exp1_confusion.png", bbox_inches="tight")
plt.show()

In [ ]:
# Ablation de Protocol : retirer Protocol dégrade-t-il le score multiclasses ?
base = results[results["task"] == "multiclass"].set_index("model")
abl = results[results["task"] == "multiclass_no_protocol"].set_index("model")
delta = (abl["macro_f1"] - base["macro_f1"]).rename("delta_macro_f1_no_protocol")
display(delta.round(4).to_frame())

base_pc = per_class[
    (per_class["task"] == "multiclass") & (per_class["model"] == "random_forest")
].set_index("class")
abl_pc = per_class[
    (per_class["task"] == "multiclass_no_protocol")
    & (per_class["model"] == "random_forest")
].set_index("class")
delta_recall = (abl_pc["recall"] - base_pc["recall"]).sort_values()
display(delta_recall.round(4).to_frame())

fig, ax = plt.subplots(figsize=(10, 5))
delta_recall.plot(kind="barh", ax=ax, color=np.where(delta_recall < 0, "#c44e52", "#55a868"))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Δ rappel (sans Protocol − avec Protocol)")
ax.set_title("Ablation de Protocol par classe — forêt aléatoire")
fig.tight_layout()
fig.savefig(FIG / "03_exp1_protocol_ablation.png", bbox_inches="tight")
plt.show()

## Bilan de H1

**Observation.** Le Macro-F1 passe de presque 1,0 en binaire à environ 0,60–0,76 pour les deux modèles multiclasses entraînés, tandis que le Weighted-F1 reste au-dessus de 0,94.  L’écart entre Macro-F1 et Weighted-F1 traduit le déséquilibre : les familles rares sont manquées alors que les grandes familles maintiennent le score pondéré.  Retirer `Protocol` change le Macro-F1 de moins de 0,01.

**Interprétation ML.** H1 est soutenue.  Un modèle peut être quasi parfait pour « attaque / pas attaque » et beaucoup plus faible pour « quelle attaque ».  Le faible écart lié à Protocol est rassurant : le modèle multiclasses ne lit pas seulement le protocole de transport, car les caractéristiques de taille de paquet et de temporalité portent l’essentiel du signal.  Le rappel par classe est la vue honnête ; la matrice de confusion 13×13 montre quelles familles sont fusionnées.

**Interprétation cyber.** Réponse à Q2 : non, un détecteur binaire n’est pas nécessairement utile pour identifier une attaque.  Un analyste a besoin du étiquette de famille pour choisir une mitigation (SYN flood vs réflexion DNS vs amplification TFTP).  Une alerte binaire est une porte de triage, pas un produit de classification.  L’ablation de Protocol suggère que le classifieur apprend la forme du flux, mais la confusion restante entre familles proches constitue la limite opérationnelle.

**Limite.** La correspondance à 13 classes fusionne `DrDoS_X` avec `X` ; cette expérience ne teste donc pas si les variantes de réflexion et d’exploitation sont séparables.  C’est le rôle plus fin de l’expérience 2.  Les données de laboratoire et une seule graine limitent la validité externe.


# 04 — Phase 4 · Expérience 2 — Leave-One-Attack-Out (H2, Q1)

## Question
Quelles familles DDoS généralisent le mieux à une famille que le modèle n’a jamais vue ?

## Hypothèse H2
La généralisation dépend de la famille laissée de côté : la plupart des mécanismes partagent assez de structure de flux pour être reconnus, mais Syn est l’exception pour la forêt aléatoire.

## Protocole
Pour chacune des 11 familles d’attaque fines comptant au moins 1 000 lignes après dédoublonnage :
1. Purger la famille laissée de côté de **tous les 17 fichiers**, pas seulement du fichier qui porte son nom.  `UDPLag-training` contient Syn et UDP ; `LDAP-training` contient NetBIOS ; `MSSQL-training` contient LDAP ; `UDP-training` contient MSSQL.  Garder une seule de ces lignes rendrait fausse l’affirmation « famille non vue ».
2. Re-découper les lignes restantes avec les graines 42, 43, 44.
3. Entraîner le binaire Benign/DDoS sur les familles restantes.
4. Tester sur les lignes de la famille laissée de côté + un échantillon Benign apparié issu du pli de test connu.  Le test est équilibré pour garder Macro-F1 et FPR interprétables.
5. Pour la graine 42, entraîner un modèle multiclasses « famille connue » plus léger et prédire les lignes laissées de côté pour remplir `confusion_to_matrix` : à quelle famille connue l’attaque non vue ressemble-t-elle ?

Exclues par la règle des 1 000 lignes : DrDoS_LDAP (993), Portmap (400), NetBIOS (374), DrDoS_NetBIOS (311), WebDDoS (49).  C’est une contrainte de support, pas un jugement sur l’importance de ces familles.

Ce notebook exécute lui-même les 99 fits LOADO et le contrôle sibling ; il écrit `results/exp2.csv` et `results/exp2_sibling_check.csv`.


In [ ]:
exp2 = run_exp2()
sibling = run_sibling(seed=42)
results = pd.read_csv(RES / "exp2.csv")
per_class = pd.read_csv(RES / "exp2_per_class.csv")
purge = pd.read_csv(RES / "exp2_purge_audit.csv")
conf_to = pd.read_csv(RES / "exp2_confusion_to.csv")

print(f"{results['family'].nunique()} familles × {results['model'].nunique()} modèles × {results['seed'].nunique()} graines = {len(results)} fits")
display(results.head())
display(purge.head(15))

In [ ]:
summary = (
    results.groupby(["family", "model"], observed=True)
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        recall_family_mean=("recall_family", "mean"),
        fpr_mean=("fpr", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
    )
    .reset_index()
)
display(summary.round(4))

fig, ax = plt.subplots(figsize=(10, 6))
pivot = summary.pivot(index="family", columns="model", values="macro_f1_mean")
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
ax.set_title("Macro-F1 LOADO par famille laissée de côté et par modèle (moyenne de 3 graines)")
ax.set_xlabel("")
ax.set_ylabel("famille laissée de côté")
fig.tight_layout()
fig.savefig(FIG / "04_exp2_family_model_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, column, title in [
    (axes[0], "recall_family_mean", "Rappel sur la famille non vue"),
    (axes[1], "fpr_mean", "Taux de faux positifs sur Benign apparié"),
]:
    pivot = summary.pivot(index="family", columns="model", values=column)
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="Blues" if column.startswith("recall") else "Reds", vmin=0, vmax=1, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("famille laissée de côté")
fig.tight_layout()
fig.savefig(FIG / "04_exp2_recall_fpr.png", bbox_inches="tight")
plt.show()

In [ ]:
# À quelle famille connue une attaque non vue est-elle confondue ?
matrix = (
    conf_to.groupby(["held_out_family", "predicted_known_family"], observed=True)["count"]
    .sum()
    .unstack(fill_value=0)
)
rates = matrix.div(matrix.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(rates, annot=True, fmt=".2f", cmap="magma_r", ax=ax)
ax.set_title("Matrice de confusion vers les familles connues : famille non vue (lignes) vs famille connue prédite (colonnes)")
ax.set_xlabel("famille connue prédite")
ax.set_ylabel("famille laissée de côté")
fig.tight_layout()
fig.savefig(FIG / "04_exp2_confusion_to.png", bbox_inches="tight")
plt.show()

display(rates.round(3))

In [ ]:
# Audit de purge : une famille n’est pas confinée au fichier qui porte son nom.
purge_matrix = purge.pivot_table(
    index="family", columns="src_file", values="rows", aggfunc="sum", fill_value=0
)
fig, ax = plt.subplots(figsize=(15, 6))
sns.heatmap(np.log1p(purge_matrix), annot=purge_matrix, fmt="d", cmap="viridis", cbar_kws={"label": "log1p(lignes purgées)"}, ax=ax)
ax.set_title("Lignes purgées par fichier source pour chaque famille laissée de côté")
ax.set_xlabel("fichier source")
ax.set_ylabel("famille laissée de côté")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIG / "04_exp2_purge_audit.png", bbox_inches="tight")
plt.show()

files_per_family = (purge_matrix > 0).sum(axis=1).sort_values(ascending=False)
display(files_per_family.rename("source_files_containing_family").to_frame())
print("Syn apparaît dans :", purge.loc[purge.family == 'Syn', 'src_file'].tolist())
print("UDP apparaît dans :", purge.loc[purge.family == 'UDP', 'src_file'].tolist())
print("LDAP apparaît dans :", purge.loc[purge.family == 'LDAP', 'src_file'].tolist())

In [ ]:
# Profil d’erreur : meilleures et pires familles pour le modèle le plus fort, moyennées sur les graines.
rf = summary[summary["model"] == "random_forest"].sort_values("recall_family_mean")
display(rf[["family", "macro_f1_mean", "macro_f1_std", "recall_family_mean", "fpr_mean", "pr_auc_mean"]].round(4))

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(rf["fpr_mean"], rf["recall_family_mean"], s=70, color="#4c72b0")
for row in rf.itertuples():
    ax.annotate(row.family, (row.fpr_mean, row.recall_family_mean), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("FPR sur Benign apparié")
ax.set_ylabel("rappel sur la famille non vue")
ax.set_title("Forêt aléatoire : détection vs fausses alertes par famille laissée de côté")
ax.set_xlim(-0.01, max(0.05, rf["fpr_mean"].max() * 1.1))
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIG / "04_exp2_recall_fpr_scatter.png", bbox_inches="tight")
plt.show()

## Bilan de H2 et réponse à Q1

**Observation.** La carte de chaleur est globalement élevée, mais elle n’est pas plate.  Pour la forêt aléatoire, dix familles restent au-dessus de 0,96 (neuf au-dessus de 0,99), tandis que **Syn s’effondre à ≈ 0,604** (rappel ≈ 0,30).  La régression logistique est beaucoup plus stable sur Syn (≈ 0,973).  La matrice de confusion vers les familles connues explique l’échec de la RF : les lignes Syn sont prédites comme `Benign` (8 540), `UDPLag` (7 919) ou `Portmap` (2 466), et non comme une famille d’attaque claire.  L’audit de purge confirme que Syn apparaît aussi dans `UDPLag-training` ; purger seulement `Syn-training` aurait donc laissé fuiter des lignes.

**Interprétation ML.** H2 est soutenue, mais de façon asymétrique : la dépendance à la famille laissée de côté est réelle et concentrée sur Syn, et elle dépend du modèle.  La RF apprend une frontière qui traite les flux de type Syn comme Benign/UDPLag, tandis que le modèle linéaire garde une frontière plus lisse qui déclenche encore.  Un F1 agrégé unique masquerait cet échec ; le tableau par famille et la matrice de confusion vers les familles connues l’exposent.

**Interprétation cyber.** Réponse à Q1 : la plupart des familles non vues restent détectables, mais Syn est l’exception pour la RF.  Les familles qui généralisent le mieux partagent un profil d’en-tête/fenêtre et de débit avec des familles vues à l’entraînement ; le motif TCP de Syn est assez proche de Benign/UDPLag pour que la RF le manque.  Pour un SOC, le cas dangereux n’est pas une mauvaise étiquette de famille mais une alerte manquée : un SYN flood non vu pourrait passer pour du bénin.  La matrice de confusion vers les familles connues est plus actionnable qu’un F1 unique, car elle indique quel comportement connu le trafic non vu imite.

**Limite.** Onze rotations ne couvrent pas tout l’espace des attaques non vues ; les familles rares ont été exclues par manque de support.  Le test Benign apparié est équilibré, ce qui rend le FPR comparable mais non représentatif de la prévalence en production.  Le modèle d’attribution est une forêt aléatoire plus légère entraînée uniquement pour le diagnostic, pas le détecteur principal.  La capture de laboratoire et trois graines limitent la validité externe.  Le contrôle sibling (cellule `run_sibling()`) vérifie si conserver `DrDoS_X` tout en laissant `X` de côté gonfle le score ; il est rapporté séparément.


# 05 — Phase 5 · Expérience 3 — Réduction de caractéristiques (H3, Q3, Q4)

## Question
Combien de variables sont réellement nécessaires, et quel gain cela apporte-t-il en coût d’inférence ?

## Hypothèse H3
Une fraction des 77 caractéristiques conserve l’essentiel de la performance ; un ensemble plus petit réduit aussi la latence d’inférence.

## Protocole
- Même split sans fuite, graines 42/43/44, entraînement plafonné à 20 000 lignes par classe pour le temps d’exécution.
- Sélection ajustée **uniquement dans le pli d’entraînement** avec `mutual_info_classif` ; l’importance par impureté de la forêt aléatoire sert de contre-vérification.
- Variantes : toutes les 77 caractéristiques, ~75 % (58), ~50 % (39), ~25 % (20), Top-10.  Les colonnes constantes sont retirées sur le pli d’entraînement ; le nombre effectif est donc reporté à côté du nombre demandé.
- Pour chaque variante : Macro-F1 et latence d’inférence.  Latence = médiane de cinq appels `predict` sur un lot de 1 000 lignes, prétraitement inclus.
- Ce notebook exécute lui-même les 30 fits et écrit `results/exp3.csv` ; il n’y a ni module ni CLI.


In [ ]:
exp3 = run_exp3()
results = pd.read_csv(RES / "exp3.csv")
ranking = pd.read_csv(RES / "exp3_feature_ranking.csv")
display(results.head())
display(results.groupby(["task", "variant"]).size())

In [ ]:
order = ["all", "75pct", "50pct", "25pct", "top10"]
agg = (
    results.groupby(["task", "variant", "n_features_effective"], observed=True)
    .agg(macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"), latency_med=("latency_ms_1000", "median"), latency_std=("latency_ms_1000", "std"))
    .reset_index()
)
agg["variant"] = pd.Categorical(agg["variant"], categories=order, ordered=True)
agg = agg.sort_values(["task", "variant"])
display(agg.round(4))

def knee_point(x, y):
    """Point de distance perpendiculaire maximale à la corde formée par les extrémités."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = (x - x.min()) / max(x.max() - x.min(), 1e-12)
    y = (y - y.min()) / max(y.max() - y.min(), 1e-12)
    p0 = np.array([x[0], y[0]])
    p1 = np.array([x[-1], y[-1]])
    line = p1 - p0
    # Produit vectoriel 2D explicite : np.cross n’accepte plus
    # les vecteurs 2D dans NumPy 2.x.
    distances = np.abs(
        line[0] * (y - p0[1]) - line[1] * (x - p0[0])
    ) / max(np.linalg.norm(line), 1e-12)
    return int(np.argmax(distances))

fig, ax = plt.subplots(figsize=(10, 5.5))
for task, color in [("binary", "#4c72b0"), ("multiclass", "#c44e52")]:
    part = agg[agg["task"] == task].sort_values("n_features_effective")
    ax.errorbar(part["n_features_effective"], part["macro_f1_mean"], yerr=part["macro_f1_std"], marker="o", capsize=3, label=task, color=color)
    k = knee_point(part["n_features_effective"], part["macro_f1_mean"])
    ax.axvline(part["n_features_effective"].iloc[k], color=color, linestyle="--", alpha=0.35)
    ax.annotate("coude", (part["n_features_effective"].iloc[k], part["macro_f1_mean"].iloc[k]), xytext=(6, -18), textcoords="offset points", color=color, fontsize=9)
ax.set_xlabel("nombre effectif de caractéristiques")
ax.set_ylabel("Macro-F1 (moyenne ± écart-type sur 3 graines)")
ax.set_title("H3 : performance vs nombre de caractéristiques")
ax.set_ylim(0, 1.02)
ax.legend(title="")
fig.tight_layout()
fig.savefig(FIG / "05_exp3_performance.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for task, color in [("binary", "#4c72b0"), ("multiclass", "#c44e52")]:
    part = agg[agg["task"] == task].sort_values("n_features_effective")
    ax.errorbar(part["n_features_effective"], part["latency_med"], yerr=part["latency_std"], marker="o", capsize=3, label=task, color=color)
    k = knee_point(part["n_features_effective"], part["latency_med"])
    ax.axvline(part["n_features_effective"].iloc[k], color=color, linestyle="--", alpha=0.35)
    ax.annotate("coude", (part["n_features_effective"].iloc[k], part["latency_med"].iloc[k]), xytext=(6, 8), textcoords="offset points", color=color, fontsize=9)
ax.set_xlabel("nombre effectif de caractéristiques")
ax.set_ylabel("latence (ms pour 1 000 flux, médiane de 5)")
ax.set_title("Q4 : coût d’inférence vs nombre de caractéristiques")
ax.legend(title="")
fig.tight_layout()
fig.savefig(FIG / "05_exp3_latency.png", bbox_inches="tight")
plt.show()

display(agg.pivot(index="variant", columns="task", values="latency_med").round(2))

In [ ]:
# Contre-vérification : information mutuelle vs importance de la forêt aléatoire.
focus = ranking[(ranking["seed"] == ranking["seed"].min()) & (ranking["task"] == "multiclass")].copy()
rho, p = spearmanr(focus["mutual_info"], focus["rf_importance"])
print(f"rho de Spearman = {rho:.3f} (p={p:.2e})")

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(focus["mutual_info"], focus["rf_importance"], s=25, alpha=0.7, color="#4c72b0")
for row in focus.nlargest(8, "mutual_info").itertuples():
    ax.annotate(row.feature, (row.mutual_info, row.rf_importance), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("information mutuelle")
ax.set_ylabel("importance de la forêt aléatoire")
ax.set_title("Les deux critères de sélection concordent largement")
fig.tight_layout()
fig.savefig(FIG / "05_exp3_feature_agreement.png", bbox_inches="tight")
plt.show()

display(focus.nlargest(15, "mutual_info")[["feature", "mutual_info", "rf_importance", "mi_rank", "rf_rank"]].round(4))

## Bilan de H3, Q3 et Q4

**Observation.** La courbe de performance n’est pas une ligne droite descendante : un sous-ensemble de caractéristiques conserve l’essentiel du Macro-F1, et c’est la variante Top-10 qui rend la baisse visible.  La courbe de latence est plus plate que celle de performance pour la forêt aléatoire : retirer des caractéristiques réduit davantage le temps d’entraînement et la mémoire que le temps de prédiction, car l’essentiel du coût est la traversée des 300 arbres.

**Interprétation ML.** H3 est partiellement soutenue.  L’information mutuelle et l’importance de la forêt aléatoire classent les mêmes caractéristiques de taille de paquet et de débit en tête ; le résultat n’est donc pas un artefact d’un seul sélecteur.  Le coude marque le point où les caractéristiques supplémentaires cessent d’apporter beaucoup de performance ; en dessous, les familles rares perdent du rappel en premier.  Réponse à Q3 : quelques dizaines de caractéristiques suffisent pour l’essentiel de la performance, mais le nombre exact dépend de la cible binaire ou à 13 classes.

**Interprétation cyber.** Réponse à Q4 : pour un usage temps réel, les chiffres importants sont le rappel sur la classe attaque, le FPR sur Benign et la latence par lot, pas l’accuracy.  Un modèle Top-10 est attractif pour un capteur en périphérie, mais il doit être validé sur les familles qui perdent du rappel.  Une forêt aléatoire à 300 arbres peut rester trop lente pour une inspection à pleine vitesse ; la courbe de réduction indique où s’arrêter, pas que le modèle est déployable.

**Limite.** La latence est mesurée sur un seul ordinateur portable, par lots, avec un cache chaud ; ce n’est pas un test de référence sur équipement réseau.  L’entraînement a été plafonné à 20 000 lignes par classe, donc les scores absolus ne sont pas directement comparables aux expériences 1 et 2.  Le sélecteur est une information mutuelle univariée ; une sélection multivariée pourrait trouver un ensemble plus petit à score égal.


# 06 — Phase 6 · Explicabilité et analyse d’erreurs

## Question
Quelles caractéristiques de flux guident les décisions, et à quoi ressemblent les faux positifs et faux négatifs pour un analyste ?

## Méthode
- `permutation_importance` sur le **pli de test**, jamais sur le pli d’entraînement.
- Une exécution diagnostique pour l’expérience 1 (forêt aléatoire à 13 classes) et une pour l’expérience 2 (trois familles laissées de côté : meilleure, médiane et pire des résultats LOADO).
- Les modèles sont réentraînés ici avec un plafond de 10 000 lignes par classe pour que le notebook reste exécutable ; les scores principaux restent ceux des expériences sur données complètes.
- Les galeries FP/FN montrent les valeurs réelles des flux mal classés.
- SHAP est optionnel dans le plan.  Il n’est pas installé sur Python 3.14 ici ; l’importance par permutation satisfait déjà l’exigence d’interprétation.

Chaque affirmation est étiquetée **Observation**, **Interprétation ML**, **Interprétation cyber** ou **Limite**.


In [ ]:
X, y_family, y_binary, meta = load_deduped()
X_train, X_test, y_family_train, y_family_test = split(X, y_family, seed=42)
print(X.shape, X_train.shape, X_test.shape)

GALLERY_COLUMNS = [
    "Protocol",
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Packet Length Mean",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "SYN Flag Count",
    "ACK Flag Count",
    "Init Fwd Win Bytes",
]

In [ ]:
# Modèle diagnostique E1 : forêt aléatoire à 13 classes sur un pli d’entraînement plafonné.
y_train = group_attack_families(y_family_train)
y_test = group_attack_families(y_family_test)
X_train_cap, y_train_cap = subsample(X_train, y_train, cap=10_000, seed=42)

e1 = build_pipeline(
    RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
)
e1.fit(X_train_cap, y_train_cap)

# Échantillon de test stratifié pour l’importance par permutation.
sample_parts = []
for _, group in X_test.assign(_y=y_test).groupby("_y", sort=False):
    sample_parts.append(group.sample(n=min(250, len(group)), random_state=42))
perm_sample = pd.concat(sample_parts).drop(columns="_y")
y_perm = y_test.loc[perm_sample.index]

perm = permutation_importance(
    e1,
    perm_sample,
    y_perm,
    scoring="f1_macro",
    n_repeats=3,
    random_state=42,
    n_jobs=1,
)
e1_perm = (
    pd.DataFrame(
        {
            "feature": perm_sample.columns,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
e1_perm.to_csv(RES / "xai_permutation_e1.csv", index=False)
display(e1_perm.head(20).round(4))

fig, ax = plt.subplots(figsize=(10, 6))
top = e1_perm.head(20).sort_values("importance_mean")
ax.barh(top["feature"], top["importance_mean"], xerr=top["importance_std"], color="#4c72b0")
ax.set_xlabel("baisse moyenne du Macro-F1 après permutation")
ax.set_title("E1 — importance par permutation sur le pli de test (RF à 13 classes)")
fig.tight_layout()
fig.savefig(FIG / "06_xai_permutation_e1.png", bbox_inches="tight")
plt.show()

In [ ]:
def balanced_eval(family, cap=10_000, seed=42):
    X_pool, y_pool, held_X, held_y = purge_family(X, y_family, family)
    X_tr, X_te, y_tr, y_te = split(X_pool, y_pool, seed=seed)
    X_tr, y_tr = subsample(X_tr, y_tr, cap=cap, seed=seed)
    benign_idx = y_te.index[y_te == BENIGN].to_numpy()
    n = min(len(held_X), len(benign_idx))
    rng = np.random.default_rng(seed)
    held_pos = rng.choice(len(held_X), size=n, replace=False) if len(held_X) > n else np.arange(len(held_X))
    ben_pos = rng.choice(benign_idx, size=n, replace=False) if len(benign_idx) > n else benign_idx
    X_attack = held_X.iloc[held_pos].reset_index(drop=True).copy()
    X_benign = X_te.loc[ben_pos].reset_index(drop=True).copy()
    X_attack.attrs = {}
    X_benign.attrs = {}
    X_eval = pd.concat([X_attack, X_benign], ignore_index=True)
    y_eval = np.r_[np.ones(len(X_attack), dtype=np.int8), np.zeros(len(X_benign), dtype=np.int8)]
    return X_tr, (y_tr != BENIGN).astype("int8"), X_eval, y_eval, X_attack

# Choix des familles laissées de côté (meilleure / médiane / pire) pour la RF à partir d’exp2.
exp2 = pd.read_csv(RES / "exp2.csv")
rf = exp2[exp2["model"] == "random_forest"].groupby("family")["recall_family"].mean().sort_values()
chosen = [rf.index[-1], rf.index[len(rf) // 2], rf.index[0]]
print("Familles E2 (meilleure, médiane, pire) :", chosen)

perm_rows = []
for family in chosen:
    X_tr, y_tr, X_eval, y_eval, _ = balanced_eval(family)
    model = build_pipeline(
        RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
    )
    model.fit(X_tr, y_tr)
    n = min(500, len(X_eval))
    idx = np.random.default_rng(42).choice(len(X_eval), size=n, replace=False)
    perm = permutation_importance(
        model,
        X_eval.iloc[idx],
        y_eval[idx],
        scoring="f1_macro",
        n_repeats=2,
        random_state=42,
        n_jobs=1,
    )
    part = pd.DataFrame(
        {
            "family": family,
            "feature": X_eval.columns,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    perm_rows.append(part)

e2_perm = pd.concat(perm_rows, ignore_index=True)
e2_perm.to_csv(RES / "xai_permutation_e2.csv", index=False)
display(e2_perm.sort_values(["family", "importance_mean"], ascending=[True, False]).groupby("family").head(10).round(4))

fig, ax = plt.subplots(figsize=(10, 5.5))
top_features = e2_perm.groupby("feature")["importance_mean"].mean().nlargest(12).index
heat = e2_perm[e2_perm["feature"].isin(top_features)].pivot(index="feature", columns="family", values="importance_mean")
sns.heatmap(heat.loc[top_features], annot=True, fmt=".3f", cmap="Blues", ax=ax)
ax.set_title("E2 — importance par permutation sur les familles non vues")
ax.set_xlabel("famille laissée de côté")
fig.tight_layout()
fig.savefig(FIG / "06_xai_permutation_e2.png", bbox_inches="tight")
plt.show()

In [ ]:
# Galerie FP/FN pour la tâche binaire E1.
y_bin_train = (y_family_train != BENIGN).astype("int8")
y_bin_test = (y_family_test != BENIGN).astype("int8")
X_bin_train, y_bin_train_cap = subsample(X_train, y_bin_train, cap=10_000, seed=42)
binary_model = build_pipeline(
    RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
)
binary_model.fit(X_bin_train, y_bin_train_cap)
pred = binary_model.predict(X_test)
proba = binary_model.predict_proba(X_test)[:, 1]
report = X_test.loc[:, GALLERY_COLUMNS].copy()
report.insert(0, "true", y_bin_test.map({0: "Benign", 1: "DDoS"}).to_numpy())
report.insert(1, "predicted", pd.Series(pred, index=report.index).map({0: "Benign", 1: "DDoS"}))
report.insert(2, "p_DDoS", proba)
fp_all = report[(report["true"] == "Benign") & (report["predicted"] == "DDoS")]
fn_all = report[(report["true"] == "DDoS") & (report["predicted"] == "Benign")]
fp = fp_all.nlargest(10, "p_DDoS")
fn = fn_all.nsmallest(10, "p_DDoS")
print(f"FP={len(fp_all):,}  FN={len(fn_all):,} sur {len(report):,} lignes de test")
print("Faux positifs (Benign classés DDoS) :")
display(fp)
print("Faux négatifs (DDoS manqués) :")
display(fn)
pd.concat([fp.assign(gallery="FP"), fn.assign(gallery="FN")]).to_csv(RES / "xai_fp_fn_e1.csv", index=False)

In [ ]:
# Galerie FP/FN pour une famille non vue (la pire d’exp2).
family = chosen[-1]
X_tr, y_tr, X_eval, y_eval, X_attack = balanced_eval(family)
model = build_pipeline(
    RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
)
model.fit(X_tr, y_tr)
pred = model.predict(X_eval)
proba = model.predict_proba(X_eval)[:, 1]
report = X_eval.loc[:, GALLERY_COLUMNS].copy()
report.insert(0, "true", pd.Series(y_eval).map({0: "Benign", 1: family}))
report.insert(1, "predicted", pd.Series(pred).map({0: "Benign", 1: "DDoS"}))
report.insert(2, "p_DDoS", proba)
print(f"famille laissée de côté : {family}")
print("Attaques manquées (FN) :")
display(report[(report['true'] == family) & (report['predicted'] == 'Benign')].nsmallest(10, 'p_DDoS'))
print("Fausses alertes Benign (FP) :")
display(report[(report['true'] == 'Benign') & (report['predicted'] == 'DDoS')].nlargest(10, 'p_DDoS'))
report.to_csv(RES / f"xai_fp_fn_e2_{family}.csv", index=False)

## Constats

**Observation.** Les principales caractéristiques de permutation E1 sont des champs d’en-tête/fenêtre TCP (`Bwd Header Length`, `Init Fwd Win Bytes`, `Init Bwd Win Bytes`) et `Flow Bytes/s`, pas les tailles moyennes de paquet ni les colonnes de débit global.  Sur E2, les familles quasi parfaites (`DrDoS_SNMP`, `TFTP`) ont une importance quasi nulle pour chaque caractéristique prise isolément : aucune caractéristique unique n’est critique ; Syn est plus fragile (`ACK Flag Count`, `Idle Max`, `Init Fwd Win Bytes`).

**Interprétation ML.** Les modèles n’utilisent pas seulement `Protocol` ; ils combinent les informations d’en-tête/fenêtre TCP avec le débit.  Cela explique à la fois le succès binaire et la confusion multiclasses entre familles ayant des profils d’en-tête similaires.  L’importance par permutation sur le test confirme que le signal n’est pas un artefact du pli d’entraînement.

**Interprétation cyber.** Les galeries FP/FN montrent ce qu’un analyste SOC voit réellement : un petit nombre de flux bénins avec des profils d’en-tête/débit proches d’attaques, et des attaques dont le débit est sous le seuil appris.  La galerie plaide pour un réglage du seuil et des procédures spécifiques à chaque famille plutôt qu’un seuil global unique.  Une famille non vue manquée est plus dangereuse qu’une famille connue mal étiquetée, car aucune règle existante ne la couvre.

**Limite.** L’importance par permutation dépend du modèle et de l’échantillon de test, et les caractéristiques corrélées se partagent le crédit.  Les modèles diagnostiques sont plafonnés à 10 000 lignes par classe ; leurs importances sont donc indicatives, pas exactes pour les modèles sur données complètes.  SHAP n’était pas installé sur Python 3.14 ; aucune conclusion n’en dépend.  Les galeries FP/FN sont des échantillons, pas une taxonomie complète des erreurs.
